In [1]:
#!/usr/bin/env python3
"""
ROBUSTNESS STUDY — QPA + EBA Tamper Sweep
==========================================
Extended analysis of how share tampering affects:
  (a) QPA pixel-precise tamper detection rate
  (b) EBA MSB-change detection rate
  (c) G1 and G2 reconstruction quality (PSNR / SSIM)
  (d) Asymmetry: S1-only vs S3-only vs all-shares tampering

Tampering sweep:
  Percentages : [0, 10, 20, 30, 50, 70] %
  Modes       : msb (flip bit7) | lsb (flip bit0) | random (random bit) | full (XOR 0xFF)
  Targets     : S1_only | S3_only | all

Runs entirely classically on the share arrays saved by encoding.
(Quantum circuit already validated decode correctness in decoding_verification.ipynb)
No quantum re-simulation needed for the sweep.

All results saved to CSV + paper-quality plots.
"""


'\nROBUSTNESS STUDY — QPA + EBA Tamper Sweep\n==========================================\nExtended analysis of how share tampering affects:\n  (a) QPA pixel-precise tamper detection rate\n  (b) EBA MSB-change detection rate\n  (c) G1 and G2 reconstruction quality (PSNR / SSIM)\n  (d) Asymmetry: S1-only vs S3-only vs all-shares tampering\n\nTampering sweep:\n  Percentages : [0, 10, 20, 30, 50, 70] %\n  Modes       : msb (flip bit7) | lsb (flip bit0) | random (random bit) | full (XOR 0xFF)\n  Targets     : S1_only | S3_only | all\n\nRuns entirely classically on the share arrays saved by encoding.\n(Quantum circuit already validated decode correctness in decoding_verification.ipynb)\nNo quantum re-simulation needed for the sweep.\n\nAll results saved to CSV + paper-quality plots.\n'

In [2]:
import os, sys, json, time
import numpy as np
import cv2
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import pandas as pd

sys.path.insert(0, os.path.dirname(os.path.abspath("__file__")))
from qvss_utils import (
    classical_parity, image_entropy, calc_metrics,
    classical_decode, extract_secret_nibble,
    compare_parity_maps, compare_eba_maps,
    apply_tampering,
)

# ═══════════════════════════════════════════════════════════════════
# CONFIGURATION
# ═══════════════════════════════════════════════════════════════════
SHARE_DIR = 'neqr_shares_eba'   # same as encoding output
OUT_DIR   = 'neqr_robustness_bigimages'
ROWS, COLS = 32, 32
BITS       = 8

# Image set to use for robustness study
IMAGE_SET_LABEL = 'A'

TAMPER_PERCENTAGES = [0, 10, 20, 30, 50, 70]
TAMPER_MODES       = ['msb', 'lsb', 'random', 'full']
SHARE_TARGETS      = ['S1_only', 'S3_only', 'all']

os.makedirs(OUT_DIR, exist_ok=True)
n_conditions = (len(TAMPER_PERCENTAGES)-1) * len(TAMPER_MODES) * len(SHARE_TARGETS) + 1
print(f"[Config]  ROWS={ROWS}  COLS={COLS}  SHARE_DIR={SHARE_DIR}")
print(f"  Sweep: {len(TAMPER_PERCENTAGES)} levels × {len(TAMPER_MODES)} modes × "
      f"{len(SHARE_TARGETS)} targets + 1 baseline = {n_conditions} conditions")


[Config]  ROWS=32  COLS=32  SHARE_DIR=neqr_shares_eba
  Sweep: 6 levels × 4 modes × 3 targets + 1 baseline = 61 conditions


In [3]:
# ── Load shares ───────────────────────────────────────────────────────
set_dir = os.path.join(SHARE_DIR, f'set_{IMAGE_SET_LABEL}')
print(f"\n[Loading set '{IMAGE_SET_LABEL}' from {set_dir}]")

shares      = {}
parity_maps = {}
eba_maps    = {}

for name in ('S1', 'S2', 'S3'):
    shares[name]      = np.load(os.path.join(set_dir, f'{name}.npy'))
    parity_maps[name] = np.load(os.path.join(set_dir, f'parity_map_{name}.npy'))
    eba_maps[name]    = np.load(os.path.join(set_dir, f'eba_map_{name}.npy')).astype(int)
    print(f"  {name}: {shares[name].shape}  min={shares[name].min()}  max={shares[name].max()}")

originals = {}
for i in range(1, 3):
    p = os.path.join(set_dir, f'G{i}_original.png')
    img = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
    if img is not None:
        originals[f'G{i}'] = img
        print(f"  G{i} original loaded")

# Baseline recovery (no tampering)
R1_base, R2_base = classical_decode(shares, ROWS, COLS)
G1_base, G2_base = extract_secret_nibble(R1_base, R2_base)
G1_orig = originals.get('G1', G1_base)
G2_orig = originals.get('G2', G2_base)
G1_msb_orig = ((G1_orig >> 4) << 4).astype(np.uint8)
G2_msb_orig = ((G2_orig >> 4) << 4).astype(np.uint8)
print(f"\nBaseline recovery done.")



[Loading set 'A' from neqr_shares_eba\set_A]
  S1: (32, 32)  min=17  max=238
  S2: (32, 32)  min=27  max=206
  S3: (32, 32)  min=32  max=255
  G1 original loaded
  G2 original loaded

Baseline recovery done.


In [4]:
# ── QPA detection function ────────────────────────────────────────────
def qpa_detect(original_parity, tampered_shares, rows, cols):
    """
    Classical QPA tamper detection:
    Compare encode-time parity map vs parity of tampered shares.
    Returns detection stats per share.
    """
    results = {}
    for sname in ['S1','S2','S3']:
        enc_p = np.array(original_parity[sname])
        dec_p = np.vectorize(classical_parity)(tampered_shares[sname]).astype(int)
        valid  = enc_p >= 0
        det    = valid & (enc_p != dec_p)
        results[sname] = {
            'detected':  int(det.sum()),
            'total':     int(valid.sum()),
            'det_pct':   100.0 * det.sum() / max(1, valid.sum()),
            'tamper_map': det.astype(np.uint8),
        }
    return results


def eba_detect(original_eba_maps, tampered_shares):
    """
    EBA MSB-change detection:
    Compare encode-time MSB map vs MSB of tampered shares.
    Detects bit-7 level tampering. Full-byte and MSB-flip modes detected at 100%.
    """
    results = {}
    for sname in ['S1','S2','S3']:
        enc_msb = original_eba_maps[sname]
        dec_msb = ((tampered_shares[sname] >> 7) & 1).astype(int)
        changed = (enc_msb != dec_msb).astype(np.uint8)
        results[sname] = {
            'detected': int(changed.sum()),
            'total':    ROWS * COLS,
            'det_pct':  100.0 * changed.sum() / (ROWS * COLS),
            'change_map': changed,
        }
    return results


In [5]:
# ── Run sweep ─────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("ROBUSTNESS SWEEP")
print("=" * 70)

rows_list = []
t_start   = time.time()

for mode in TAMPER_MODES:
    for target in SHARE_TARGETS:
        for pct in TAMPER_PERCENTAGES:
            if pct == 0:
                t_shares = shares
                t_masks  = {k: np.zeros((ROWS,COLS), bool) for k in ['S1','S2','S3']}
                label    = 'baseline'
            else:
                t_shares, t_masks = apply_tampering(
                    shares, pct, mode, target, ROWS, COLS, seed=42)
                label = f'{mode}_{target}_{pct}pct'

            R1t, R2t   = classical_decode(t_shares, ROWS, COLS)
            G1t, G2t   = extract_secret_nibble(R1t, R2t)

            # Recovery quality
            m1 = calc_metrics(G1_msb_orig, G1t)
            m2 = calc_metrics(G2_msb_orig, G2t)

            # QPA detection
            qpa_res = qpa_detect(parity_maps, t_shares, ROWS, COLS)

            # EBA MSB detection
            eba_res = eba_detect(eba_maps, t_shares)

            # Actual tamper pixel count
            actual_tampered = {k: int(t_masks[k].sum()) for k in ['S1','S2','S3']}

            row = {
                'tamper_pct':   pct,
                'tamper_mode':  mode if pct > 0 else 'none',
                'share_target': target if pct > 0 else 'none',
                # G1 metrics
                'G1_PSNR':  m1['PSNR'],
                'G1_SSIM':  m1['SSIM'],
                'G1_MSE':   m1['MSE'],
                'G1_px_mm': m1['px_mismatch'],
                # G2 metrics
                'G2_PSNR':  m2['PSNR'],
                'G2_SSIM':  m2['SSIM'],
                'G2_MSE':   m2['MSE'],
                'G2_px_mm': m2['px_mismatch'],
                # QPA detection per share
                'S1_qpa_det_pct': qpa_res['S1']['det_pct'],
                'S2_qpa_det_pct': qpa_res['S2']['det_pct'],
                'S3_qpa_det_pct': qpa_res['S3']['det_pct'],
                'S1_qpa_detected': qpa_res['S1']['detected'],
                'S2_qpa_detected': qpa_res['S2']['detected'],
                'S3_qpa_detected': qpa_res['S3']['detected'],
                # EBA MSB detection per share
                'S1_eba_det_pct': eba_res['S1']['det_pct'],
                'S2_eba_det_pct': eba_res['S2']['det_pct'],
                'S3_eba_det_pct': eba_res['S3']['det_pct'],
                # Actual tampered pixels
                'S1_actual_tampered': actual_tampered['S1'],
                'S2_actual_tampered': actual_tampered['S2'],
                'S3_actual_tampered': actual_tampered['S3'],
            }
            rows_list.append(row)

            if pct > 0:
                print(f"  {mode:<7} {target:<10} {pct:3d}%  "
                      f"G1_PSNR={m1['PSNR']:.2f}  G2_PSNR={m2['PSNR']:.2f}  "
                      f"QPA_S1={qpa_res['S1']['det_pct']:.1f}%  "
                      f"EBA_S1={eba_res['S1']['det_pct']:.1f}%")

print(f"\nSweep done in {time.time()-t_start:.1f}s  ({len(rows_list)} conditions)")

df = pd.DataFrame(rows_list)
csv_path = os.path.join(OUT_DIR, 'robustness_results.csv')
df.to_csv(csv_path, index=False)
print(f"Results saved → {csv_path}")



ROBUSTNESS SWEEP
  msb     S1_only     10%  G1_PSNR=15.68  G2_PSNR=15.68  QPA_S1=10.7%  EBA_S1=10.7%
  msb     S1_only     20%  G1_PSNR=12.49  G2_PSNR=12.49  QPA_S1=22.4%  EBA_S1=22.4%
  msb     S1_only     30%  G1_PSNR=10.96  G2_PSNR=10.96  QPA_S1=31.8%  EBA_S1=31.8%
  msb     S1_only     50%  G1_PSNR=9.01  G2_PSNR=9.01  QPA_S1=49.9%  EBA_S1=49.9%
  msb     S1_only     70%  G1_PSNR=7.49  G2_PSNR=7.49  QPA_S1=70.8%  EBA_S1=70.8%
  msb     S3_only     10%  G1_PSNR=inf  G2_PSNR=inf  QPA_S1=0.0%  EBA_S1=0.0%
  msb     S3_only     20%  G1_PSNR=inf  G2_PSNR=inf  QPA_S1=0.0%  EBA_S1=0.0%
  msb     S3_only     30%  G1_PSNR=inf  G2_PSNR=inf  QPA_S1=0.0%  EBA_S1=0.0%
  msb     S3_only     50%  G1_PSNR=inf  G2_PSNR=inf  QPA_S1=0.0%  EBA_S1=0.0%
  msb     S3_only     70%  G1_PSNR=inf  G2_PSNR=inf  QPA_S1=0.0%  EBA_S1=0.0%
  msb     all         10%  G1_PSNR=15.68  G2_PSNR=13.04  QPA_S1=10.7%  EBA_S1=10.7%
  msb     all         20%  G1_PSNR=12.49  G2_PSNR=10.87  QPA_S1=22.4%  EBA_S1=22.4%
  msb   

In [6]:
# ── Plot 1: PSNR degradation vs tampering % ───────────────────────────
BG='#0d1117'; PANEL='#161b22'; TEXT='#e6edf3'; GRID='#30363d'
C_G1='#448aff'; C_G2='#ff9800'
MODE_COLS = {'msb': '#4fc3f7', 'lsb': '#00e676', 'random': '#ffd600', 'full': '#ff1744'}
MODE_LABELS = {'msb': 'MSB-only', 'lsb': 'LSB-only', 'random': 'Random bit', 'full': 'Full byte'}
LS = {'S1_only': '-', 'S3_only': '--', 'all': ':'}
MARK = {'S1_only': 'o', 'S3_only': 's', 'all': '^'}
PCTS = [p for p in TAMPER_PERCENTAGES if p > 0]

def sax(ax, title):
    ax.set_facecolor(PANEL)
    ax.tick_params(colors=TEXT, labelsize=9)
    for sp in ax.spines.values(): sp.set_color(GRID)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    ax.grid(color=GRID, linestyle='--', alpha=0.5)
    ax.set_title(title, color=TEXT, fontsize=11, fontweight='bold')

fig, axes = plt.subplots(len(TAMPER_MODES), 2, figsize=(16, 4*len(TAMPER_MODES)))
fig.patch.set_facecolor(BG)

for row_idx, mode in enumerate(TAMPER_MODES):
    for col_idx, (secret, col) in enumerate([('G1', C_G1), ('G2', C_G2)]):
        ax = axes[row_idx, col_idx]
        sax(ax, f'{secret} PSNR — {MODE_LABELS[mode]}')
        for target in SHARE_TARGETS:
            sub = df[(df['tamper_mode']==mode) & (df['share_target']==target)]
            sub = sub[sub['tamper_pct']>0].sort_values('tamper_pct')
            psnr_col = f'{secret}_PSNR'
            vals = [min(v, 45) for v in sub[psnr_col].tolist()]
            ax.plot(sub['tamper_pct'].tolist(), vals,
                    color=MODE_COLS[mode], linestyle=LS[target], marker=MARK[target],
                    linewidth=2, markersize=7, label=target)
        ax.set_xlabel('Tampering (%)', color=TEXT)
        ax.set_ylabel('PSNR (dB)', color=TEXT)
        ax.legend(facecolor=PANEL, edgecolor=GRID, labelcolor=TEXT, fontsize=8)

fig.suptitle('PSNR Degradation vs Tampering Percentage\nRows: mode | Cols: G1/G2 recovery',
             color=TEXT, fontsize=13, fontweight='bold')
plt.tight_layout(pad=1.5)
out = os.path.join(OUT_DIR, 'psnr_degradation.png')
plt.savefig(out, dpi=180, bbox_inches='tight', facecolor=BG)
plt.close(); print(f"PSNR plot saved → {out}")


PSNR plot saved → neqr_robustness_bigimages\psnr_degradation.png


In [7]:
# ── Plot 2: QPA detection rate ────────────────────────────────────────
fig, axes = plt.subplots(len(TAMPER_MODES), 3, figsize=(18, 4*len(TAMPER_MODES)))
fig.patch.set_facecolor(BG)

for row_idx, mode in enumerate(TAMPER_MODES):
    for col_idx, sname in enumerate(['S1','S2','S3']):
        ax = axes[row_idx, col_idx]
        sax(ax, f'QPA {sname} — {MODE_LABELS[mode]}')
        for target in SHARE_TARGETS:
            sub = df[(df['tamper_mode']==mode) & (df['share_target']==target)]
            sub = sub[sub['tamper_pct']>0].sort_values('tamper_pct')
            ax.plot(sub['tamper_pct'].tolist(),
                    sub[f'{sname}_qpa_det_pct'].tolist(),
                    color=MODE_COLS[mode], linestyle=LS[target], marker=MARK[target],
                    linewidth=2, markersize=7, label=target)
        ax.axhline(95, color='#ffd600', linestyle=':', linewidth=1.5, label='95% threshold')
        ax.set_ylim(0, 108)
        ax.set_xlabel('Tampering (%)', color=TEXT)
        ax.set_ylabel('Detection Rate (%)', color=TEXT)
        ax.legend(facecolor=PANEL, edgecolor=GRID, labelcolor=TEXT, fontsize=8)

fig.suptitle('QPA Pixel-Precise Tamper Detection Rate\nRows: mode | Cols: S1/S2/S3',
             color=TEXT, fontsize=13, fontweight='bold')
plt.tight_layout(pad=1.5)
out = os.path.join(OUT_DIR, 'qpa_detection_rate.png')
plt.savefig(out, dpi=180, bbox_inches='tight', facecolor=BG)
plt.close(); print(f"QPA detection plot saved → {out}")


QPA detection plot saved → neqr_robustness_bigimages\qpa_detection_rate.png


In [8]:
# ── Plot 3: EBA MSB detection rate ────────────────────────────────────
fig, axes = plt.subplots(len(TAMPER_MODES), 3, figsize=(18, 4*len(TAMPER_MODES)))
fig.patch.set_facecolor(BG)

for row_idx, mode in enumerate(TAMPER_MODES):
    for col_idx, sname in enumerate(['S1','S2','S3']):
        ax = axes[row_idx, col_idx]
        sax(ax, f'EBA MSB {sname} — {MODE_LABELS[mode]}')
        for target in SHARE_TARGETS:
            sub = df[(df['tamper_mode']==mode) & (df['share_target']==target)]
            sub = sub[sub['tamper_pct']>0].sort_values('tamper_pct')
            ax.plot(sub['tamper_pct'].tolist(),
                    sub[f'{sname}_eba_det_pct'].tolist(),
                    color=MODE_COLS[mode], linestyle=LS[target], marker=MARK[target],
                    linewidth=2, markersize=7, label=target)
        ax.set_ylim(0, 108)
        ax.set_xlabel('Tampering (%)', color=TEXT)
        ax.set_ylabel('EBA MSB Change Rate (%)', color=TEXT)
        ax.legend(facecolor=PANEL, edgecolor=GRID, labelcolor=TEXT, fontsize=8)

fig.suptitle('EBA MSB-Change Detection Rate vs Tampering\n'
             'MSB=bit7: full/msb modes→100%, lsb→0%, random→12.5%',
             color=TEXT, fontsize=13, fontweight='bold')
plt.tight_layout(pad=1.5)
out = os.path.join(OUT_DIR, 'eba_detection_rate.png')
plt.savefig(out, dpi=180, bbox_inches='tight', facecolor=BG)
plt.close(); print(f"EBA detection plot saved → {out}")


EBA detection plot saved → neqr_robustness_bigimages\eba_detection_rate.png


In [9]:
# ── Plot 4: G1 vs G2 recovery asymmetry ──────────────────────────────
fig, axes = plt.subplots(2, len(TAMPER_MODES), figsize=(5*len(TAMPER_MODES), 10))
fig.patch.set_facecolor(BG)

for col_idx, mode in enumerate(TAMPER_MODES):
    for row_idx, (secret, col) in enumerate([('G1', C_G1), ('G2', C_G2)]):
        ax = axes[row_idx, col_idx]
        sax(ax, f'{secret} — {MODE_LABELS[mode]}')
        for target in SHARE_TARGETS:
            sub = df[(df['tamper_mode']==mode) & (df['share_target']==target)]
            sub = sub[sub['tamper_pct']>0].sort_values('tamper_pct')
            vals = [min(v, 45) for v in sub[f'{secret}_PSNR'].tolist()]
            ax.plot(sub['tamper_pct'].tolist(), vals,
                    color=MODE_COLS[mode], linestyle=LS[target], marker=MARK[target],
                    linewidth=2.2, markersize=8, label=target)
        ax.set_xlabel('Tampering (%)', color=TEXT)
        ax.set_ylabel('PSNR (dB)', color=TEXT)
        ax.legend(facecolor=PANEL, edgecolor=GRID, labelcolor=TEXT, fontsize=8)

fig.suptitle('G1 vs G2 Recovery Asymmetry — Code 1 Nibble Architecture\n'
             'S3-only tampering degrades G2 but NOT G1 (structural property)',
             color=TEXT, fontsize=13, fontweight='bold')
plt.tight_layout(pad=1.5)
out = os.path.join(OUT_DIR, 'g1_g2_asymmetry.png')
plt.savefig(out, dpi=180, bbox_inches='tight', facecolor=BG)
plt.close(); print(f"Asymmetry plot saved → {out}")


Asymmetry plot saved → neqr_robustness_bigimages\g1_g2_asymmetry.png


In [10]:
# ── Summary table ─────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("ROBUSTNESS SUMMARY — full byte tampering, S1-only target")
print("=" * 70)
summary = df[
    ((df['tamper_mode']=='full') & (df['share_target']=='S1_only')) |
    (df['tamper_mode']=='none')
][[
    'tamper_pct', 'tamper_mode',
    'G1_PSNR', 'G1_SSIM', 'G2_PSNR', 'G2_SSIM',
    'S1_qpa_det_pct', 'S1_eba_det_pct',
]].copy()
summary['G1_PSNR'] = summary['G1_PSNR'].apply(lambda x: '+inf' if x==float('inf') else f'{x:.2f}')
summary['G2_PSNR'] = summary['G2_PSNR'].apply(lambda x: '+inf' if x==float('inf') else f'{x:.2f}')
print(summary.to_string(index=False))

print(f"\nAll robustness outputs saved to {OUT_DIR}/")
print("DONE.")



ROBUSTNESS SUMMARY — full byte tampering, S1-only target
 tamper_pct tamper_mode G1_PSNR   G1_SSIM G2_PSNR   G2_SSIM  S1_qpa_det_pct  S1_eba_det_pct
          0        none    +inf  1.000000    +inf  1.000000             0.0        0.000000
          0        none    +inf  1.000000    +inf  1.000000             0.0        0.000000
          0        none    +inf  1.000000    +inf  1.000000             0.0        0.000000
          0        none    +inf  1.000000    +inf  1.000000             0.0        0.000000
          0        none    +inf  1.000000    +inf  1.000000             0.0        0.000000
          0        none    +inf  1.000000    +inf  1.000000             0.0        0.000000
          0        none    +inf  1.000000    +inf  1.000000             0.0        0.000000
          0        none    +inf  1.000000    +inf  1.000000             0.0        0.000000
          0        none    +inf  1.000000    +inf  1.000000             0.0        0.000000
          0        non